In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)

## load dataset:

In [54]:
# 1.
load=pd.read_csv('clean_repair_cost_predict.csv')

In [55]:
load['failure_flag']=load[['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF']].max(axis=1)

In [56]:
load.drop(['Product ID','Machine failure','TWF','HDF','PWF','OSF','RNF','UDI'],axis=1,inplace=True)

In [57]:
load=pd.get_dummies(load,drop_first=True).astype(int)

In [ ]:
load.columns


In [ ]:
load['failure_flag'].value_counts()

______________________________________________________________________________________________________________________________________________

## model creation and DS concepts:

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [7]:
from sklearn.metrics import classification_report,confusion_matrix

In [58]:
# 2.
x=load.drop('failure_flag',axis=1)
y=load['failure_flag']

In [ ]:
x_train.shape

In [59]:
# 3.
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.75,random_state=42)

### Tuning  rf model:

In [ ]:
rf=RandomForestClassifier(class_weight='balanced',random_state=42)

In [ ]:
model_pred=rf.fit(xtrain,ytrain).predict(xtest)

In [ ]:
cm=confusion_matrix(ytest,model_pred)

### model accuracy:

In [ ]:
cm


In [ ]:
cls=classification_report(ytest,model_pred)

In [ ]:
print(cls)


______________________________________________________________________________________________________________________________________________

In [ ]:
______________________________________________________________________________________________________________________________________________

In [ ]:
data=load['failure_flag'].value_counts()

In [ ]:
plt.pie(data.values,labels=['not failure','failure'],autopct="%1.1f%%",colors=['green','red'])
plt.show()

______________________________________________________________________________________________________________________________________________

# handle imbalanced data:


## over sampling:

In [10]:
from imblearn.over_sampling import RandomOverSampler

In [60]:
# 4.
over=RandomOverSampler()
x_re,y_re=over.fit_resample(x_train,y_train)
rf=RandomForestClassifier()
fit=rf.fit(x_re,y_re)
y_pred=fit.predict(x_test)

## oversample accuracy:

In [ ]:
cm=confusion_matrix(ytest,y_pred)
cm

In [ ]:
y_re.shape

In [ ]:
# cls=classification_report(ytest,ypred)
# print(cls)

------------------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------------------

# Feature Selection:

In [13]:
from sklearn.feature_selection import SelectKBest,chi2

In [63]:
def features(xtrain,ytrain,xtest,n):
    kb=SelectKBest(score_func=chi2,k=n)
    x_train_fe=kb.fit_transform(xtrain,ytrain)
    x_test_fe=kb.transform(xtest)
    mask=kb.get_support()
    selected_features=xtrain.columns[mask]
    
    return x_train_fe,x_test_fe,selected_features

In [15]:
from sklearn.metrics import accuracy_score
def acc(classifier,ytest,xtest):
    ypred=classifier.predict(xtest)
    ac=accuracy_score(ytest,ypred)
    cm,report=clf(ytest,ypred)
    return ac,cm,report
    

In [16]:
def clf(ytest,ypred):
    cm=confusion_matrix(ytest,ypred)
    clfs=classification_report(ytest,ypred)
    return cm,clfs

In [17]:
def model_creation(model,xtrain,ytrain,xtest,ytest):
    classifier=model.fit(xtrain,ytrain)
    accuracy,cm,rep=acc(classifier,ytest,xtest)
    return round(accuracy,2),cm,rep
    
    

In [18]:
# import models:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [19]:
dt=DecisionTreeClassifier()
rt=RandomForestClassifier()
svm=SVC()
xg=XGBClassifier()

In [20]:
models=[dt,rt,svm,xg]

In [22]:
# mask=kb.get_support()
# cols=x_re.columns[mask]
# cols

In [ ]:
kb.scores_

In [64]:
xtrain,xtest,features=features(x_re,y_re,x_test,5)

In [65]:
score=[]
for model in models:
    accs,cm,clf_rep=model_creation(model,xtrain,y_re,xtest,y_test)
    score.append(accs)
    print(f'{model} : {accs}')
    print(cm)
    print()

DecisionTreeClassifier() : 0.96
[[2373   49]
 [  52   26]]

RandomForestClassifier() : 0.97
[[2391   31]
 [  47   31]]

SVC() : 0.83
[[2016  406]
 [  20   58]]

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...) : 0.95
[[2343   79]
 [  38   40]]



In [ ]:
score

In [66]:
features

Index(['Air temperature [K]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L',
       'Type_M'],
      dtype='object')